<div style="
    background: linear-gradient(135deg, #102542 0%, #0F4C5C 45%, #1A759F 100%);
    padding: 34px;
    border-radius: 14px;
    margin: 10px 0 20px 0;
    box-shadow: 0 10px 26px rgba(16,37,66,0.34);
    border: 1px solid rgba(255,255,255,0.12);
">
    <h1 style="
        color: #ffffff;
        margin: 0;
        text-align: center;
        font-size: 34px;
        font-weight: 300;
        letter-spacing: 2px;
    ">PHY-3500 · Notebook 02</h1>
    <p style="
        color: rgba(255,255,255,0.92);
        margin: 10px 0 0 0;
        text-align: center;
        font-size: 18px;
        letter-spacing: 0.5px;
    ">UMAP et t-SNE sur les scores PCA</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.14), rgba(26,117,159,0.12));
    border: 1px solid rgba(15,76,92,0.30);
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<strong>Équipe :</strong> Alex, Justine<br>
<strong>Cours :</strong> PHY-3500 Physique numérique - Université Laval<br>
<strong>Prérequis :</strong> Notebook 01 (01_pca.ipynb) exécuté pour charger les scores PCA
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.14), rgba(16,37,66,0.14));
    border: 1px solid rgba(26,117,159,0.30);
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<h3 style="margin: 0 0 10px 0; color: #0F4C5C;">Objectifs</h3>
<ol style="margin: 0; padding-left: 20px; line-height: 1.6;">
  <li>Calculer des embeddings UMAP et t-SNE à partir des scores PCA.</li>
  <li>Visualiser la structure 2D avec colorations physiques (T_eff, log g, [Fe/H], BP-RP).</li>
  <li>Évaluer la stabilité via Procrustes multi-seeds.</li>
  <li>Vérifier un contrôle négatif sur des données permutées.</li>
  <li>Explorer la sensibilité aux hyperparamètres.</li>
  <li>Projeter les embeddings sur le diagramme HR.</li>
</ol>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 14px 16px;
    margin-top: 12px;
">
<strong>Note méthodologique :</strong> UMAP/t-SNE sont appliqués sur les scores PCA (et non sur X brut) pour réduire le bruit, accélérer le calcul et stabiliser la topologie des embeddings.
</div>

<div style="
    background: linear-gradient(135deg, rgba(16,37,66,0.10), rgba(26,117,159,0.10));
    border-left: 5px solid #1A759F;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 10px;
">
<strong>Lecture scientifique :</strong> ce notebook compare deux méthodes non linéaires complémentaires.
UMAP préserve mieux les structures globales du graphe de voisinage, tandis que t-SNE optimise surtout la cohérence locale.
</div>

<a id="toc"></a>

## Table des matières

<div style="display: grid; grid-template-columns: repeat(2, minmax(260px, 1fr)); gap: 12px; margin: 14px 0 22px 0;">

  <div style="background: linear-gradient(135deg, #0F4C5C, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#setup" style="color: white; text-decoration: none;"><strong>0. Imports et chargement</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#umap" style="color: white; text-decoration: none;"><strong>1. Embedding UMAP</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #0F4C5C, #134B5F); padding: 14px; border-radius: 10px;">
    <a href="#tsne" style="color: white; text-decoration: none;"><strong>2. Embedding t-SNE</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #134B5F, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#stability" style="color: white; text-decoration: none;"><strong>3. Stabilité (Procrustes)</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#negative" style="color: white; text-decoration: none;"><strong>4. Contrôle négatif</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #0F4C5C, #134B5F); padding: 14px; border-radius: 10px;">
    <a href="#sensitivity" style="color: white; text-decoration: none;"><strong>5. Sensibilité hyperparamètres</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #134B5F, #1A759F); padding: 14px; border-radius: 10px;">
    <a href="#hr" style="color: white; text-decoration: none;"><strong>6. Diagramme HR</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #1A759F, #0F4C5C); padding: 14px; border-radius: 10px;">
    <a href="#save" style="color: white; text-decoration: none;"><strong>7. Sauvegarde des embeddings</strong></a>
  </div>

</div>

<a id="setup"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">0 · Imports et chargement des scores PCA</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Cette cellule initialise l’environnement, charge les artefacts du NB01 et fixe les chemins de sortie.
On garantit ainsi une reproductibilité stricte (même dataset, même scores PCA, même seed).
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# ── Environnement projet (même pattern que 00_master_pipeline) ─────────────
sys.path.insert(0, str(Path("../..").resolve() / "src"))

from utils import setup_project_env

paths = setup_project_env(verbose=True)

# ── Chemins ────────────────────────────────────────────────────────────────
PCA_OUTPUT  = Path(paths["REPORTS_DIR"]) / "phy3500_pca_output.joblib"

if not PCA_OUTPUT.exists():
    raise FileNotFoundError(
        f"Fichier PCA introuvable : {PCA_OUTPUT}\n"
        "→ Lancer d'abord 01_pca.ipynb (cellule 11 — Sauvegarde)."
    )

# ── Imports dimred ──────────────────────────────────────────────────────────
from dimred import EmbeddingEngine, DimRedVisualizer
from dimred.embedding import compare_embeddings
from dimred.dimred_visualizer import CLASS_COLORS, CLASS_MARKERS

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
RANDOM_STATE = 42

# ── Chargement des scores PCA ──────────────────────────────────────────────
pca_output  = joblib.load(PCA_OUTPUT)
scores_95   = pca_output["scores_95pct"]   # Features (95% variance)
y           = pca_output["y"]
meta        = pca_output["meta"]
n_pcs       = pca_output["n_components_95pct"]

# ── Dossier figures (sous-dossier par fichier de features) ─────────────────
FIGURES_ROOT = Path(paths["REPORTS_DIR"]) / "figures" / "phy3500"
features_stem = pca_output.get("features_stem", "unknown")
FIGURES_DIR   = FIGURES_ROOT / features_stem
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nScores PCA chargés : {pca_output['scores'].shape}")
print(f"Scores 95% ({n_pcs} PCs) : {scores_95.shape}")
print(f"Classes : {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Figures : {FIGURES_DIR}")

viz = DimRedVisualizer(figsize=(9, 7), dpi=150, output_dir=FIGURES_DIR)

<a id="umap"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #134B5F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1 · Embedding UMAP</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Principe :</strong> UMAP construit un graphe de voisinage local (k-NN), puis cherche une projection 2D qui préserve au mieux la structure de connectivité probabiliste.
</div>

La fonction objectif de UMAP peut être vue comme une cross-entropie entre les similarités haute dimension et basse dimension :

$$
\mathcal{L}_{\mathrm{UMAP}} = \sum_{i,j}\left[w_{ij}\log\left(\frac{w_{ij}}{\tilde w_{ij}}\right) + (1-w_{ij})\log\left(\frac{1-w_{ij}}{1-\tilde w_{ij}}\right)\right]
$$

- `n_neighbors` contrôle l’échelle locale de la géométrie (petit = très local, grand = plus global).
- `min_dist` contrôle la compacité des amas en 2D (petit = groupes plus serrés).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
umap_engine = EmbeddingEngine(
    method="umap",
    n_components=2,
    random_state=RANDOM_STATE,
    n_neighbors=15,
    min_dist=0.1,
)

Z_umap = umap_engine.fit_transform(scores_95)
print(f"UMAP terminé en {umap_engine.fit_time_:.1f}s | shape : {Z_umap.shape}")

In [ ]:
# Coloration par classe
fig, ax = viz.plot_embedding(
    Z_umap, y=y,
    title="UMAP — Spectres LAMOST DR5 (n_neighbors=15, min_dist=0.1)",
    method="UMAP", s=2, alpha=0.45,
    save_path=FIGURES_DIR / "umap_classes.png",
)
plt.show()

In [ ]:
# ============================================================================
# UMAP zoom + comparaison de classes (2 paires de figures, chaque paire ensemble)
# Pair 1: STAR + GALAXY + QSO (full + zoom dans UNE meme figure)
# Pair 2: GALAXY + QSO (STAR filtree) (full + zoom dans UNE meme figure)
# ============================================================================

# >>> MODIFIER ICI : FENETRE DE ZOOM <<<
ZOOM_X = (1.0, 4.0)  # ex: (2.5, 4.5)
ZOOM_Y = (5.5, 8.0)  # ex: (2.0, 5.0)

x_min, x_max = ZOOM_X
y_min, y_max = ZOOM_Y
if not (x_min < x_max and y_min < y_max):
    raise ValueError("La fenetre de zoom est invalide: verifier ZOOM_X et ZOOM_Y.")

# Normaliser les etiquettes de classes (robuste aux variantes/typos)
labels = np.char.upper(np.asarray(y).astype(str))
labels = np.char.strip(labels)
label_aliases = {
    "GLAXAXY": "GALAXY",
    "GALAXIE": "GALAXY",
    "QS0": "QSO",
}
for bad, good in label_aliases.items():
    labels = np.where(labels == bad, good, labels)

# Styles visuels par classe
styles = {
    "STAR": {
        "color": CLASS_COLORS.get("STAR", "#4C72B0"),
        "marker": CLASS_MARKERS.get("STAR", "o"),
        "size_full": 2,
        "alpha_full": 0.22,
        "size_zoom": 6,
        "alpha_zoom": 0.35,
    },
    "GALAXY": {
        "color": CLASS_COLORS.get("GALAXY", "#DD8452"),
        "marker": CLASS_MARKERS.get("GALAXY", "s"),
        "size_full": 18,
        "alpha_full": 0.85,
        "size_zoom": 42,
        "alpha_zoom": 0.95,
    },
    "QSO": {
        "color": CLASS_COLORS.get("QSO", "#55A868"),
        "marker": CLASS_MARKERS.get("QSO", "^"),
        "size_full": 22,
        "alpha_full": 0.90,
        "size_zoom": 48,
        "alpha_zoom": 0.95,
    },
}

from matplotlib.patches import Rectangle


def plot_pair_together(class_list, base_name, title_prefix):
    """Genere une figure 1x2: panneau gauche=vue complete, droite=zoom."""
    zoom_window_mask = (
        (Z_umap[:, 0] >= x_min) & (Z_umap[:, 0] <= x_max)
        & (Z_umap[:, 1] >= y_min) & (Z_umap[:, 1] <= y_max)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

    # ----- Panneau gauche : vue complete -----
    ax = axes[0]
    for cls in class_list:
        cls_mask = labels == cls
        n_cls = int(cls_mask.sum())
        if n_cls == 0:
            continue

        st = styles[cls]
        ax.scatter(
            Z_umap[cls_mask, 0],
            Z_umap[cls_mask, 1],
            c=st["color"],
            marker=st["marker"],
            s=st["size_full"],
            alpha=st["alpha_full"],
            linewidths=0,
            rasterized=True,
            label=f"{cls} (n={n_cls})",
        )

    ax.add_patch(
        Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            fill=False,
            edgecolor="#2F3E46",
            linewidth=1.8,
            linestyle="--",
        )
    )

    ax.set_title(f"{title_prefix} - vue complete", fontsize=11)
    ax.set_xlabel("UMAP axis 1")
    ax.set_ylabel("UMAP axis 2")
    ax.grid(True, alpha=0.25)

    handles_full, _ = ax.get_legend_handles_labels()
    if handles_full:
        ax.legend(framealpha=0.92, loc="best")

    # ----- Panneau droit : zoom -----
    ax = axes[1]
    total_zoom_points = 0

    for cls in class_list:
        cls_zoom_mask = (labels == cls) & zoom_window_mask
        n_zoom = int(cls_zoom_mask.sum())
        if n_zoom == 0:
            continue

        total_zoom_points += n_zoom
        st = styles[cls]
        ax.scatter(
            Z_umap[cls_zoom_mask, 0],
            Z_umap[cls_zoom_mask, 1],
            c=st["color"],
            marker=st["marker"],
            s=st["size_zoom"],
            alpha=st["alpha_zoom"],
            linewidths=0,
            label=f"{cls} (n={n_zoom})",
        )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(
        f"{title_prefix} - zoom x=[{x_min},{x_max}], y=[{y_min},{y_max}]",
        fontsize=11,
    )
    ax.set_xlabel("UMAP axis 1")
    ax.set_ylabel("UMAP axis 2")
    ax.grid(True, alpha=0.25)

    handles_zoom, _ = ax.get_legend_handles_labels()
    if handles_zoom:
        ax.legend(framealpha=0.92, loc="best")
    else:
        ax.text(
            0.5,
            0.5,
            "Aucun point pour ces classes\ndans cette fenetre de zoom",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#999999", alpha=0.9),
        )

    fig.suptitle(
        f"{title_prefix} | Comparaison vue complete vs zoom",
        fontsize=13,
        fontweight="bold",
        y=1.02,
    )
    fig.tight_layout()

    pair_path = FIGURES_DIR / f"{base_name}_pair.png"
    fig.savefig(pair_path, bbox_inches="tight", dpi=150)
    plt.show()

    print(f"Saved: {pair_path}")
    for cls in class_list:
        n_zoom = int(((labels == cls) & zoom_window_mask).sum())
        print(f"  {cls} in zoom: {n_zoom}")
    print(f"  Total in zoom: {total_zoom_points}")


# Pair 1: STAR + GALAXY + QSO ensemble (figures 1 et 2)
plot_pair_together(
    class_list=("STAR", "GALAXY", "QSO"),
    base_name="umap_all_classes",
    title_prefix="UMAP STAR + GALAXY + QSO",
)

# Pair 2: GALAXY + QSO ensemble (figures 3 et 4)
plot_pair_together(
    class_list=("GALAXY", "QSO"),
    base_name="umap_no_star",
    title_prefix="UMAP GALAXY + QSO (STAR filtree)",
)

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Structure UMAP et non-linéarité</h4>
<p style="margin: 0 0 8px 0;">L’embedding UMAP révèle des filaments et des sous-populations absents d’une projection linéaire simple. Cela indique que la variété spectrale est intrinsèquement non linéaire.</p>
<p style="margin: 0;">Une continuité de gradient en T_eff et [Fe/H] à travers ces structures suggère que la projection conserve une information astrophysique réelle et pas seulement une séparation artificielle.</p>
</div>

### Lecture scientifique rapide
- UMAP préserve bien les voisinages locaux tout en gardant une structure globale exploitable.
- La présence d’amas denses et de transitions progressives est cohérente avec des populations stellaires en continuum physique.
- Une comparaison avec le contrôle négatif est essentielle pour valider que cette géométrie n’est pas un artefact de la méthode.

In [ ]:
# Grille multi-coloration
fig, axes = viz.plot_embedding_grid(
    Z_umap, y=y, meta=meta,
    method="UMAP",
    params=["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"],
    s=2, alpha=0.4,
    save_path=FIGURES_DIR / "umap_grid.png",
)
plt.show()

<a id="hdbscan"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.2 · Clustering automatisé — HDBSCAN sur espace UMAP</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Objectif</h4>
<p style="margin: 0 0 8px 0;">
UMAP révèle une structure filamentaire riche — mais sans étiquettes automatiques. On applique <strong>HDBSCAN</strong> (Hierarchical Density-Based Spatial Clustering of Applications with Noise) directement sur les coordonnées 2D de UMAP pour <em>quantifier mathématiquement</em> les populations stellaires identifiées visuellement.
</p>
<p style="margin: 0;">
Avantage clé de HDBSCAN vs K-Means : il gère les formes arbitraires (filaments, croissants), ne force pas tous les points dans un cluster, et classe les objets atypiques comme <em>bruit</em> — ce qui en astrophysique correspond aux anomalies spectrales (QSOs, galaxies, spectres défectueux).
</p>
</div>

In [ ]:
# ── §1.2 HDBSCAN — ajustement et figures (cellules 12+13) ─────────────────
from dimred.hdbscan_analyzer import HDBSCANAnalyzer, compute_sensitivity
import matplotlib.pyplot as plt

HDBSCAN_MIN_CLUSTER = 75
HDBSCAN_MIN_SAMPLES = 20

hdb = HDBSCANAnalyzer(
    min_cluster_size=HDBSCAN_MIN_CLUSTER,
    min_samples=HDBSCAN_MIN_SAMPLES,
)
hdb.fit(Z_umap)

cluster_labels = hdb.labels_
n_clusters     = hdb.n_clusters_
n_noise        = hdb.n_noise_
cluster_ids    = hdb.cluster_ids_
color_map      = hdb.color_map_
clusterer      = hdb.clusterer_

print(f'HDBSCAN terminé :')
print(f'  Clusters détectés    : {n_clusters}')
print(f'  Points classifiés    : {(cluster_labels != -1).sum()} ({100*(cluster_labels != -1).sum()/len(cluster_labels):.1f}%)')
print(f'  Points bruit (noise) : {n_noise} ({100*n_noise/len(cluster_labels):.1f}%)')

# ── Figures ────────────────────────────────────────────────────────────────
fig, axes = viz.plot_hdbscan_clusters(Z_umap, hdb, y,
    save_path=FIGURES_DIR / 'umap_hdbscan_clusters.png')
plt.show()

fig, axes = viz.plot_hdbscan_hr(Z_umap, hdb, meta,
    save_path=FIGURES_DIR / 'umap_hdbscan_hr.png')
plt.show()

# Sensibilité
df_sens = compute_sensitivity(Z_umap, min_samples=HDBSCAN_MIN_SAMPLES)
print()
print('Sensibilité HDBSCAN :')
print(df_sens.to_string(index=False))
fig, axes = viz.plot_hdbscan_sensitivity(df_sens,
    min_cluster_val=HDBSCAN_MIN_CLUSTER, pres_cluster_val=300,
    save_path=FIGURES_DIR / 'umap_hdbscan_sensitivity.png')
plt.show()

# Version présentation
labels_pres = hdb.fit_presentation(Z_umap, min_cluster_size=300)
print(f'\nVersion présentation : {hdb.n_clusters_pres_} clusters')
fig, axes = viz.plot_hdbscan_presentation(Z_umap, hdb, meta,
    save_path=FIGURES_DIR / 'umap_hdbscan_present.png')
plt.show()


In [ ]:
# ── §1.2 (suite) Profil physique et interprétation astrophysique ──────────
df_table, df_interp = hdb.physical_profile_table(meta, y)
hdb.print_physical_profile(df_table, df_interp)


<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — HDBSCAN et découverte non supervisée de populations stellaires</h4>

<p style="margin: 0 0 8px 0;">
<strong>Ce que l'algorithme accomplit.</strong> En n'utilisant que la <em>géométrie 2D de l'espace UMAP</em> — sans aucun label, sans paramètre physique, sans connaissance astrophysique préalable — HDBSCAN identifie automatiquement des groupes distincts. Lorsqu'on croise ces groupes avec les paramètres Gaia DR3, on redécouvre les grandes familles de la séquence stellaire : naines de la séquence principale, géantes rouges, étoiles de type solaire, populations métaliquement distinctes.
</p>

<p style="margin: 0 0 8px 0;">
<strong>Le bruit comme information.</strong> Les points classés comme "bruit" par HDBSCAN ne sont pas des erreurs — ce sont les objets <em>spectroscopiquement atypiques</em> : spectres de basse qualité, binaires à raies doubles, étoiles variables, ou les rares galaxies et QSOs du catalogue. L'autoencodeur (NB03) les détecte via une MSE de reconstruction élevée ; HDBSCAN les détecte via leur isolement topologique dans l'espace UMAP. Les deux approches se confirment mutuellement.
</p>

<p style="margin: 0 0 8px 0;">
<strong>Comparaison avec XGBoost supervisé.</strong> Le pipeline AstroSpectro principal (XGBoost) atteint ~84% de balanced accuracy sur les types spectraux LAMOST. HDBSCAN, sans aucune supervision, retrouve des structures qui correspondent aux grandes classes physiques — ce qui valide que les features spectroscopiques engineerées contiennent bien le signal astrophysique nécessaire, et pas seulement du signal de classification arbitraire.
</p>

<p style="margin: 0;">
<strong>Sensibilité aux hyperparamètres.</strong> <code>min_cluster_size</code> contrôle la granularité : une valeur faible fragmente la séquence principale en sous-populations fines (branche des naines K, naines M), une valeur élevée ne retient que les grandes structures. La valeur choisie (150 sur 42 920 spectres, soit ~0.35%) offre un bon compromis entre découverte et robustesse.
</p>
</div>

<a id="hdbscan-spectres"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.3 · Profil spectroscopique moyen par cluster HDBSCAN</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0;">
HDBSCAN a identifié les clusters géométriquement dans l'espace UMAP. Cette section <em>ferme la boucle</em> : on retourne dans l'espace des features originales pour calculer le spectre moyen de chaque population. Si les clusters correspondent à des populations physiques réelles, leurs profils de features doivent être caractéristiques des types spectraux connus (A, F, G, K, M).
</p>
</div>

In [ ]:
# ── §1.3 Heatmap features × clusters HDBSCAN ──────────────────────────────
from dimred.hdbscan_analyzer import compute_feature_profiles
import matplotlib.pyplot as plt

df_feat, cluster_means, cluster_labels_aligned = compute_feature_profiles(
    paths=paths,
    features_stem=features_stem,
    Z_umap=Z_umap,
    cluster_labels=cluster_labels,
)

fig, ax = viz.plot_hdbscan_feature_heatmap(
    cluster_means, cluster_labels_aligned,
    save_path=FIGURES_DIR / 'umap_hdbscan_feature_heatmap.png',
)
plt.show()

fig, axes = viz.plot_hdbscan_feature_profiles(
    cluster_means, cluster_labels_aligned,
    save_path=FIGURES_DIR / 'umap_hdbscan_feature_profiles.png',
)
plt.show()


<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — Profils spectraux et classification MK</h4>
<p style="margin: 0 0 8px 0;">
La heatmap ferme la boucle entre la réduction de dimension et la physique stellaire. Les clusters montrant un <strong>excès de Hα_eq_width</strong> et un <strong>déficit de Mg_b / CaII</strong> correspondent aux étoiles chaudes (types A–F) — exactement l'opposition que PC1 de la PCA captait avec ρ=+0.832. Les clusters inverses (CaII fort, Hα faible) correspondent aux étoiles froides K–M.
</p>
<p style="margin: 0;">
Ce résultat montre que la classification non supervisée (UMAP → HDBSCAN) <em>redécouvre automatiquement</em> la séquence de classification spectrale MK établie au 19ème siècle — mais à partir de rien d'autre que la géométrie de 208 descripteurs spectraux. C'est la validation physique ultime du pipeline.
</p>
</div>

<a id="xgboost-umap"></a>

<div style="background: linear-gradient(135deg, #134074 0%, #3C6E71 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(19,64,116,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.4 · UMAP × XGBoost — Le pont supervisé / non-supervisé</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(19,64,116,0.12), rgba(60,110,113,0.12));
    border-left: 6px solid #134074;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0 0 8px 0;">
Le pipeline AstroSpectro (supervisé, XGBoost, ~84% balanced accuracy) et l'analyse de réduction de dimension (non supervisée, PHY-3500) ont jusqu'ici fonctionné en <em>silo</em>. Cette cellule crée le lien manquant : on applique le classificateur XGBoost entraîné aux mêmes 42 920 spectres et on colorie l'espace UMAP par les <strong>prédictions du modèle</strong>.
</p>
<p style="margin: 0;">
Questions clés : Les classes prédites par XGBoost correspondent-elles aux clusters HDBSCAN ? Où se situent les erreurs de classification (confusion F/G) dans l'espace UMAP ? Les zones d'incertitude sont-elles aux frontières entre clusters ?
</p>
</div>

In [ ]:
# ── §1.4 Pont XGBoost — prédiction sur l'espace UMAP ──────────────────────
from dimred.xgboost_bridge import load_and_predict
import matplotlib.pyplot as plt

xgb_result = load_and_predict(
    paths=paths,
    features_stem=features_stem,
    Z_umap=Z_umap,
    y=y,
    cluster_labels=cluster_labels,
    color_map=color_map,
)

y_pred          = xgb_result['y_pred']
confidence      = xgb_result['confidence']
classes_pred    = xgb_result['classes_pred']
y_aligned       = xgb_result['y_aligned']
cl_aligned      = xgb_result['cl_aligned']
Z_umap_aligned  = xgb_result['Z_umap_aligned']
fg_mask         = xgb_result['fg_mask']

if xgb_result['_xgb_ok']:
    print(f'Classes prédites : {dict(zip(*__import__("numpy").unique(y_pred, return_counts=True)))}')

    fig, axes = viz.plot_xgboost_umap(
        Z_umap_aligned, y_pred, confidence, cl_aligned, color_map,
        save_path=FIGURES_DIR / 'umap_xgboost_predictions.png',
    )
    path_xgb = FIGURES_DIR / 'umap_xgboost_predictions.png'
    plt.show()

    fig, ax = viz.plot_fg_confusion(
        Z_umap_aligned, y_pred,
        save_path=FIGURES_DIR / 'umap_xgboost_FG_confusion.png',
    )
    path_fg = FIGURES_DIR / 'umap_xgboost_FG_confusion.png'
    plt.show()
else:
    print('⚠  XGBoost indisponible — §1.4 ignorée.')
    path_xgb = path_fg = None


<div style="
    background: linear-gradient(135deg, rgba(19,64,116,0.12), rgba(60,110,113,0.12));
    border-left: 6px solid #134074;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #134074;">Interprétation — Correspondance supervisé / non-supervisé</h4>
<p style="margin: 0 0 8px 0;">
Si les clusters HDBSCAN et les prédictions XGBoost concordent largement, cela démontre que l'espace spectral contient un signal astrophysique robuste que deux approches fondamentalement différentes capturent de façon cohérente.
</p>
<p style="margin: 0 0 8px 0;">
La carte de confiance est particulièrement révélatrice : les zones de faible confiance (P_max &lt; 0.6) se concentrent aux <em>frontières entre clusters</em> HDBSCAN — là où la géométrie non-supervisée est elle-même ambiguë. Le modèle supervisé et l'algorithme non-supervisé s'accordent donc sur <em>où</em> les données sont ambiguës.
</p>
<p style="margin: 0;">
La confusion F/G est particulièrement instructive : ces étoiles occupent une région continue dans l'espace UMAP (pas de frontière nette), ce qui explique pourquoi XGBoost les confond — la séparation physique entre F et G est progressive, pas discrète.
</p>
</div>

<a id="umap3d"></a>

<div style="background: linear-gradient(135deg, #3C6E71 0%, #2D6A4F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(60,110,113,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1.5 · Variété stellaire en 3D — UMAP interactif (Plotly)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(60,110,113,0.12), rgba(45,106,79,0.12));
    border-left: 6px solid #3C6E71;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<p style="margin: 0 0 8px 0;">
UMAP 2D projette les 50 dimensions PCA en 2, mais compresse nécessairement de l'information. Une projection en <strong>3 dimensions</strong> permet de vérifier si le troisième axe encode un paramètre physique indépendant de T_eff et log g — typiquement la <em>métallicité [Fe/H]</em> ou la <em>vitesse de rotation</em>.
</p>
<p style="margin: 0;">
La figure Plotly est sauvegardée en <code>.html</code> — ouvrez-la dans un navigateur pour faire tourner la variété stellaire en 3D lors de la présentation au Musée de la Civilisation.
</p>
</div>

In [ ]:
# ── §1.5 UMAP 3D — visualisation interactive Plotly ───────────────────────

umap3d_result = umap_engine.compute_umap3d(
    scores_95=scores_95,
    meta=meta,
    y=y,
    cluster_labels=cluster_labels,
    figures_dir=FIGURES_DIR,
    random_state=RANDOM_STATE,
)

Z_umap3           = umap3d_result['Z_umap3']
axis3_best_param  = umap3d_result['axis3_best_param']
axis3_best_corr   = umap3d_result['axis3_best_corr']
path_3d_html      = umap3d_result['path_3d_html']
path_3d_teff      = umap3d_result['path_3d_teff']
path_3d_feh       = umap3d_result['path_3d_feh']

print(f'UMAP 3D — axe 3 encode principalement : {axis3_best_param} (ρ={axis3_best_corr:+.3f})')
print(f'Figures HTML exportées dans : {FIGURES_DIR}')


<div style="
    background: linear-gradient(135deg, rgba(60,110,113,0.12), rgba(45,106,79,0.12));
    border-left: 6px solid #3C6E71;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #3C6E71;">Interprétation — Ce qu'encode le 3ème axe UMAP</h4>
<p style="margin: 0 0 8px 0;">
Le tableau de corrélations Spearman révèle si le passage de 2D à 3D ajoute de l'information physique réelle ou seulement du bruit. Si l'axe 3 corrèle fortement avec [Fe/H] (et peu avec T_eff déjà capturée par les axes 1-2), cela démontre que l'espace spectral stellaire a au moins <strong>3 dimensions physiques indépendantes</strong> : température, gravité de surface, et abondance métallique.
</p>
<p style="margin: 0 0 8px 0;">
Ce résultat serait en accord avec la classification spectrale MK étendue qui distingue les populations galactiques (Pop I, Pop II, halo) par leur métallicité, et avec la trouvaille SHAP d'AstroSpectro où Ca II H&K et Mg b (indicateurs de métallicité) dominent la classification XGBoost.
</p>
<p style="margin: 0;">
<strong>Pour la présentation au Musée :</strong> faire tourner le globe 3D tout en expliquant que chaque axe correspond à un paramètre physique fondamental de l'étoile — l'audience verra littéralement la structure de l'espace stellaire en trois dimensions.
</p>
</div>

<a id="tsne"></a>

<div style="background: linear-gradient(135deg, #134B5F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">2 · Embedding t-SNE</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(19,75,95,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Principe :</strong> t-SNE transforme les distances en probabilités de voisinage et minimise une divergence de Kullback-Leibler entre l’espace d’entrée et l’espace projeté.
</div>

En haute dimension, les similarités conditionnelles sont définies par :

$$
p_{j|i} = \frac{\exp\left(-\|x_i-x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k\neq i}\exp\left(-\|x_i-x_k\|^2 / 2\sigma_i^2\right)}
$$

La projection 2D est optimisée via :

$$
\mathrm{KL}(P\|Q)=\sum_{i\neq j} p_{ij}\log\left(\frac{p_{ij}}{q_{ij}}\right)
$$

- `perplexity` règle le compromis local/global (ordre de grandeur du nombre de voisins effectifs).
- t-SNE est excellent pour la séparation locale, mais ses distances globales sont moins interprétables.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
tsne_engine = EmbeddingEngine(
    method="tsne",
    n_components=2,
    random_state=RANDOM_STATE,
    perplexity=30,
    max_iter=1000,
)

Z_tsne = tsne_engine.fit_transform(scores_95)
print(f"t-SNE terminé en {tsne_engine.fit_time_:.1f}s")

In [ ]:
fig, axes = viz.plot_embedding_grid(
    Z_tsne, y=y, meta=meta,
    method="t-SNE",
    params=["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"],
    s=2, alpha=0.4,
    save_path=FIGURES_DIR / "tsne_grid.png",
)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(19,75,95,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Lecture de la carte t-SNE</h4>
<p style="margin: 0 0 8px 0;">t-SNE sépare généralement mieux les voisinages immédiats que UMAP, ce qui le rend utile pour isoler de petits groupes et des structures fines.</p>
<p style="margin: 0;">En contrepartie, les distances entre amas ne doivent pas être interprétées comme des distances physiques globales fiables.</p>
</div>

- À privilégier pour l’exploration locale et la visualisation de sous-groupes.
- À compléter par UMAP/PCA pour une lecture plus globale de la géométrie des données.

<a id="stability"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3 · Analyse de stabilité (Procrustes)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
La stabilité est évaluée en relançant chaque méthode avec plusieurs seeds, puis en alignant les cartes obtenues par analyse de Procrustes.
</div>

Après translation, rotation et mise à l’échelle optimales, la distance de Procrustes peut se résumer par :

$$
d_{\mathrm{proc}}(A,B)=\|\hat{A}-\hat{B}\|_F
$$

- Plus $d_{proc}$ est proche de 0, plus la méthode est stable face au hasard d’initialisation.
- Cette métrique est essentielle pour juger la robustesse scientifique d’un embedding.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
print("Calcul stabilité UMAP (5 seeds)...")
stab_umap = umap_engine.stability_report(scores_95, n_seeds=5)
print(stab_umap.to_string(index=False))

fig, ax = viz.plot_stability(
    stab_umap,
    save_path=FIGURES_DIR / "stability_umap.png",
)
plt.show()

<a id="trustworthiness"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3.bis · Trustworthiness — fidélité de voisinage UMAP et t-SNE</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">La <strong>trustworthiness</strong> (Venna &amp; Oja, 2006) mesure dans quelle proportion les voisins proches dans l'espace d'entrée sont aussi proches dans l'espace réduit. Elle complète la stabilité Procrustes : où la stabilité évalue la <em>reproductibilité</em> (même résultat à chaque seed), la trustworthiness évalue la <em>fidélité</em> (les proximités sont-elles respectées).</p>
<p style="margin: 0;">Formellement, pour N points et k voisins :</p>
</div>

$$T(k) = 1 - \frac{2}{Nk(2N-3k-1)} \sum_{i=1}^{N} \sum_{j \in \mathcal{U}_k(i)} (r(i,j) - k)$$

où $\mathcal{U}_k(i)$ est l'ensemble des points qui sont parmi les $k$ plus proches voisins de $i$ dans l'espace 2D mais **pas** dans l'espace d'entrée, et $r(i,j)$ est leur rang dans l'espace d'entrée.

Un score proche de **1.0** = les voisinages sont parfaitement préservés.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── §3.bis Trustworthiness UMAP vs t-SNE ──────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = viz.plot_trustworthiness(
    X_ref=scores_95,
    Z_umap=Z_umap,
    Z_tsne=Z_tsne,
    save_path=FIGURES_DIR / 'umap_trustworthiness.png',
)
plt.show()


<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #0F4C5C;">Interprétation — Trustworthiness et choix de méthode</h4>
<p style="margin: 0 0 8px 0;">Un score élevé à <strong>petit k</strong> signifie que les voisins immédiats sont fidèlement représentés — la structure locale est préservée. Un score élevé à <strong>grand k</strong> indique que la structure globale (groupes distants) est aussi respectée.</p>
<p style="margin: 0 0 8px 0;">En général, <strong>t-SNE</strong> excelle à petit k (structure locale très fine) mais dégrade souvent à grand k. <strong>UMAP</strong> offre un meilleur compromis local/global grâce à son initialisation spectrale et son optimisation du graphe de voisinage.</p>
<p style="margin: 0;">Combiné avec la stabilité Procrustes (t-SNE plus stable, UMAP plus reproductible dans la topologie globale), ces deux métriques donnent un portrait complet et objectif des deux méthodes.</p>
</div>

<a id="negative"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">4 · Contrôle négatif (données permutées)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #1A759F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Le contrôle négatif consiste à permuter les colonnes (ou les lignes selon le protocole) pour briser les corrélations réelles tout en conservant les distributions marginales.
</div>

Interprétation attendue :
- Si la structure observée sur les données réelles disparaît après permutation, cela appuie la validité physique de l’embedding.
- Si une structure similaire persiste, il faut suspecter un artefact méthodologique.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
Z_umap_neg = umap_engine.negative_control(scores_95)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)
classes = np.unique(y)

for ax, Z_plot, title in zip(
    axes,
    [Z_umap, Z_umap_neg],
    ["UMAP — Données réelles", "UMAP — Contrôle négatif (X permuté)"],
):
    for cls in classes:
        mask = y == cls
        ax.scatter(
            Z_plot[mask, 0], Z_plot[mask, 1],
            c=CLASS_COLORS.get(cls, "gray"),
            marker=CLASS_MARKERS.get(cls, "o"),
            s=2, alpha=0.4, label=cls, rasterized=True,
        )
    ax.set_title(title, fontsize=11)
    ax.legend(markerscale=3, fontsize=8)
    ax.set_aspect("equal", "box")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

fig.suptitle("Validation : données réelles vs contrôle négatif", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "umap_negative_control.png", bbox_inches="tight", dpi=150)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #0F4C5C;">Interprétation - Contrôle négatif</h4>
<p style="margin: 0 0 8px 0;">Le contrôle négatif est une vérification critique : la permutation détruit la structure corrélée tout en conservant les distributions marginales.</p>
<p style="margin: 0;">La disparition des filaments sur les données permutées confirme que les motifs observés sur les données réelles proviennent de corrélations physiques et non d’un artefact de réduction de dimension.</p>
</div>

### Point méthode
- Cette étape joue un rôle de test de falsification.
- Elle renforce la crédibilité des interprétations astrophysiques tirées des projections UMAP/t-SNE.

<a id="sensitivity"></a>

<div style="background: linear-gradient(135deg, #134B5F 0%, #1A759F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(26,117,159,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5 · Sensibilité aux hyperparamètres</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(26,117,159,0.12), rgba(19,75,95,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Cette section teste la robustesse qualitative des cartes face aux paramètres clés : `n_neighbors` (UMAP) et `perplexity` (t-SNE).
</div>

### Lecture scientifique
- Une méthode robuste conserve les grands motifs d’organisation quand on fait varier les hyperparamètres dans une plage raisonnable.
- Des changements trop brutaux signalent une dépendance excessive au réglage et limitent la reproductibilité.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #1A759F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# UMAP : variation de n_neighbors
sensitivity_umap = umap_engine.parameter_sensitivity(
    scores_95,
    param_grid={"n_neighbors": [5, 15, 30, 50, 100]},
)

fig, axes = plt.subplots(1, len(sensitivity_umap),
                         figsize=(5 * len(sensitivity_umap), 5), dpi=120)

for ax, (label, Z_s) in zip(axes, sensitivity_umap.items()):
    for cls in classes:
        mask = y == cls
        ax.scatter(Z_s[mask, 0], Z_s[mask, 1],
                   c=CLASS_COLORS.get(cls, "gray"),
                   marker=CLASS_MARKERS.get(cls, "o"),
                   s=1.5, alpha=0.35, rasterized=True)
    ax.set_title(f"UMAP\n{label}", fontsize=10)
    ax.set_aspect("equal", "box")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Sensibilité UMAP — variation de n_neighbors", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "umap_sensitivity_n_neighbors.png", bbox_inches="tight", dpi=120)
plt.show()

In [ ]:
# t-SNE : variation de perplexity
sensitivity_tsne = tsne_engine.parameter_sensitivity(
    scores_95,
    param_grid={"perplexity": [5, 15, 30, 50, 100]},
)

fig, axes = plt.subplots(1, len(sensitivity_tsne),
                         figsize=(5 * len(sensitivity_tsne), 5), dpi=120)

for ax, (label, Z_s) in zip(axes, sensitivity_tsne.items()):
    for cls in classes:
        mask = y == cls
        ax.scatter(Z_s[mask, 0], Z_s[mask, 1],
                   c=CLASS_COLORS.get(cls, "gray"),
                   marker=CLASS_MARKERS.get(cls, "o"),
                   s=1.5, alpha=0.35, rasterized=True)
    ax.set_title(f"t-SNE\n{label}", fontsize=10)
    ax.set_aspect("equal", "box")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Sensibilité t-SNE — variation de perplexity", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "tsne_sensitivity_perplexity.png", bbox_inches="tight", dpi=120)
plt.show()

<a id="hr"></a>

<div style="background: linear-gradient(135deg, #1A759F 0%, #0F4C5C 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">6 · Diagramme HR coloré par coordonnée d’embedding</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(15,76,92,0.12), rgba(26,117,159,0.12));
    border-left: 6px solid #0F4C5C;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Projeter les coordonnées d’embedding sur le diagramme HR permet de vérifier si les axes appris suivent des gradients astrophysiques plausibles (température, évolution stellaire, métallicité).
</div>

Rappel (forme simplifiée) pour la magnitude absolue en bande G :

$$
M_G \approx G + 5\log_{10}(\varpi_{\mathrm{mas}}) - 10
$$

- Une cohérence visuelle entre structures HR et structures UMAP/t-SNE renforce l’interprétation physique.
- Cette étape relie directement la géométrie mathématique des embeddings à la physique stellaire.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_umap, component=0, method="UMAP",
    save_path=FIGURES_DIR / "hr_diagram_umap_ax1.png",
)
plt.show()

fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_umap, component=1, method="UMAP",
    save_path=FIGURES_DIR / "hr_diagram_umap_ax2.png",
)
plt.show()

fig, ax = viz.plot_hr_diagram_embedding(
    meta=meta, Z=Z_tsne, component=0, method="t-SNE",
    save_path=FIGURES_DIR / "hr_diagram_tsne_ax1.png",
)
plt.show()

<a id="save"></a>

<div style="background: linear-gradient(135deg, #0F4C5C 0%, #134B5F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(15,76,92,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">7 · Sauvegarde des embeddings et rapports de run</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(19,75,95,0.12), rgba(15,76,92,0.12));
    border-left: 6px solid #134B5F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
Cette cellule exporte les artefacts du run (joblib, JSON, TXT) avec horodatage et version latest pour assurer la traçabilité expérimentale.
</div>

### Pourquoi c’est important
- Reproductibilité : mêmes paramètres, mêmes entrées, mêmes sorties.
- Audit scientifique : suivi des temps, de la stabilité, des hyperparamètres et des figures générées.
- Réutilisation : les embeddings sont immédiatement disponibles pour NB03 et le rapport final.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #0F4C5C; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Sauvegarde du run NB02 ─────────────────────────────────────────────
from dimred.run_reporter import save_umap_tsne_run

result = save_umap_tsne_run(
    Z_umap=Z_umap   if "Z_umap"   in dir() else None,
    Z_tsne=Z_tsne   if "Z_tsne"   in dir() else None,
    Z_umap3=Z_umap3 if "Z_umap3"  in dir() else None,
    Z_umap_neg=Z_umap_neg if "Z_umap_neg" in dir() else None,
    y=y,
    meta=meta,
    scores_95=scores_95 if "scores_95" in dir() else None,
    umap_engine=umap_engine if "umap_engine" in dir() else None,
    tsne_engine=tsne_engine if "tsne_engine" in dir() else None,
    paths=paths,
    FIGURES_DIR=FIGURES_DIR,
    pca_output_path=str(PCA_OUTPUT) if "PCA_OUTPUT" in dir() else None,
    # Sections optionnelles
    cluster_labels=cluster_labels if "cluster_labels" in dir() else None,
    labels_pres=labels_pres       if "labels_pres"   in dir() else None,
    clusterer=clusterer           if "clusterer"     in dir() else None,
    y_pred=y_pred                 if "y_pred"        in dir() else None,
    confidence=confidence         if "confidence"    in dir() else None,
    classes_pred=classes_pred     if "classes_pred"  in dir() else None,
    y_aligned=y_aligned           if "y_aligned"     in dir() else None,
    cl_aligned=cl_aligned         if "cl_aligned"    in dir() else None,
    fg_mask=fg_mask               if "fg_mask"       in dir() else None,
    path_xgb=path_xgb             if "path_xgb"      in dir() else None,
    path_fg=path_fg               if "path_fg"       in dir() else None,
    axis3_best_param=axis3_best_param if "axis3_best_param" in dir() else None,
    axis3_best_corr=axis3_best_corr   if "axis3_best_corr"  in dir() else None,
    path_3d_html=path_3d_html     if "path_3d_html"  in dir() else None,
    path_3d_teff=path_3d_teff     if "path_3d_teff"  in dir() else None,
    path_3d_feh=path_3d_feh       if "path_3d_feh"   in dir() else None,
    sensitivity_umap=sensitivity_umap if "sensitivity_umap" in dir() else None,
    sensitivity_tsne=sensitivity_tsne if "sensitivity_tsne" in dir() else None,
    stab_umap=stab_umap if "stab_umap" in dir() else None,
    stab_tsne=stab_tsne if "stab_tsne" in dir() else None,
    ZOOM_X=ZOOM_X if "ZOOM_X" in dir() else None,
    ZOOM_Y=ZOOM_Y if "ZOOM_Y" in dir() else None,
    features_stem=features_stem if "features_stem" in dir() else None,
)

print(result["summary"])

<div style="
    background: linear-gradient(135deg, #102542 0%, #0F4C5C 50%, #1A759F 100%);
    padding: 18px 22px;
    border-radius: 12px;
    margin: 18px 0 12px 0;
    box-shadow: 0 8px 24px rgba(16,37,66,0.30);
">
  <h2 style="color: white; margin: 0; font-weight: 350; letter-spacing: 1px;">Résumé des résultats</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 12px;
">
Tableau complété à partir du run <strong>20260405T214233Z</strong>
(<code>data/reports/runs/phy3500_umap_tsne/phy3500_umap_tsne_run_20260405T214233Z.json</code>).
</div>

| Analyse | Résultat clé (run 20260405T214233Z) |
|---------|--------------------------------------|
| Échantillons utilisés | 43 019 objets (91 scores PCA, mode features) |
| Répartition des classes | STAR=42 956, GALAXY=56, QSO=7 |
| UMAP (n_neighbors=15, min_dist=0.1) | temps = 40.13 s ; shape = (43 019, 2) |
| t-SNE (perplexity=30) | temps = 80.20 s ; shape = (43 019, 2) |
| Stabilité moyenne UMAP (Procrustes) | 0.0239 (variabilité modérée entre seeds) |
| Stabilité moyenne t-SNE (Procrustes) | 0.000379 (très stable dans ce protocole) |
| Contrôle négatif UMAP | structure contractée (σx=2.200, σy=1.750) vs réel (σx=3.932, σy=4.541) |
| Sensibilité hyperparamètres | motifs globaux conservés pour UMAP et t-SNE sur les grilles testées |
| HDBSCAN (min_cluster_size=75) | 20 clusters ; 6.14% de bruit (2 643 / 43 019) |
| UMAP 3D (axe 3) | corrélation principale avec log g (ρ=-0.214) |

### Lecture rapide pour le rapport
- UMAP offre un excellent compromis entre lisibilité globale, gradients physiques et coût de calcul sur 43k objets.
- t-SNE fournit une séparation locale très fine, avec une stabilité seed-to-seed exceptionnellement élevée dans ce protocole.
- Le contrôle négatif confirme que les structures observées sur les données réelles portent une information corrélée astrophysique.
- Le clustering HDBSCAN met en évidence 20 populations avec une fraction de bruit limitée (~6%), cohérente avec une structure non supervisée riche.

<div style="
    margin-top: 14px;
    padding: 12px 14px;
    border-left: 5px solid #0F4C5C;
    background: linear-gradient(135deg, rgba(15,76,92,0.10), rgba(26,117,159,0.10));
    border-radius: 8px;
">
  <strong>Notebook suivant :</strong> <a href="./phy3500_03_autoencoder.ipynb">phy3500_03_autoencoder.ipynb</a>
</div>

<div style="
    background: linear-gradient(135deg, #134B5F 0%, #1A759F 100%);
    padding: 16px 20px;
    border-radius: 10px;
    margin: 18px 0 10px 0;
    box-shadow: 0 6px 18px rgba(19,75,95,0.25);
">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 0.8px;">Références scientifiques et techniques</h2>
</div>

### UMAP et apprentissage de variétés
1. McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426. (fondements théoriques et algorithmiques de UMAP)
2. Narayan, A., Berger, B., & Cho, H. (2021). *Assessing single-cell transcriptomic variability through density-preserving data visualization*. Nature Biotechnology, 39, 765-774. (discussion pratique sur les propriétés de visualisation de type UMAP)

### t-SNE et voisinages probabilistes
3. van der Maaten, L., & Hinton, G. (2008). *Visualizing Data using t-SNE*. Journal of Machine Learning Research, 9, 2579-2605. (article fondateur t-SNE)
4. van der Maaten, L. (2014). *Accelerating t-SNE using Tree-Based Algorithms*. JMLR, 15, 3221-3245. (aspects numériques et scalabilité)

### Stabilité, alignement et validation
5. Gower, J. C. (1975). *Generalized Procrustes Analysis*. Psychometrika, 40(1), 33-51. (base de l’alignement Procrustes)
6. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning* (chapitres sur l’évaluation et la robustesse des représentations). MIT Press. (cadre général sur stabilité des représentations)

### Données astrophysiques et contexte instrumental
7. Gaia Collaboration (2023). *Gaia Data Release 3: Summary of the content and survey properties*. Astronomy & Astrophysics, 674, A1. (paramètres astrométriques et photométriques utilisés)
8. Luo, A.-L., et al. (2019). *The LAMOST DR5 release*. Research in Astronomy and Astrophysics. (contexte instrumental LAMOST et produits spectraux)

### Implémentation et bonnes pratiques Python
9. scikit-learn developers. *TSNE documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
10. umap-learn developers. *UMAP documentation*. https://umap-learn.readthedocs.io/

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 12px;
">
Ces références couvrent la théorie des embeddings non linéaires, les méthodes de validation (stabilité/contrôle négatif), le contexte astrophysique (Gaia/LAMOST) et les détails d’implémentation.
</div>